In [1]:
# Clear any existing GPU memory
import torch
import gc

torch.cuda.empty_cache()
gc.collect()

# Check available memory
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"Available: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9:.2f} GB")

GPU Memory: 15.83 GB
Available: 15.83 GB


In [ ]:
# RUN THIS ONCE, THEN RESTART THE SESSION AND THEN DO NOT RUN THIS CELL AGAIN
# Removed bitsandbytes as we're not using quantization
!pip install -q -U accelerate transformers flask flask-cors pyngrok

In [ ]:
import torch
import re
import unicodedata
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import threading

In [4]:
# PASTE YOUR NGROK TOKEN BELOW inside the quotes
NGROK_AUTH_TOKEN = "36ZSxCZUYnZVSpKzp3tMXXVhhKW_5dcPZBWDWP83woV5LSDVE"
MODEL_ID = "large-traversaal/Alif-1.0-8B-Instruct"

In [ ]:
print("⏳ Loading Model... this may take a few minutes...")

# Load model without quantization (full precision)
# Note: This requires more GPU memory but provides better quality
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16  # Use float16 for memory efficiency while maintaining quality
)


⏳ Loading Model... this may take a few minutes...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# Initialize Pipeline
chatbot = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto", return_full_text=False)
print("✅ Model Loaded Successfully!")


In [ ]:
# ==========================================
# FLASK API SETUP
# ==========================================
app = Flask(__name__)
CORS(app)  # Enable Cross-Origin Resource Sharing for your local frontend


In [ ]:
# Default prompt (fallback only - prompts are now provided from frontend)
# This is used only if no custom_prompt is sent from the frontend
STATIC_PROMPT_CONTEXT = """You are an expert Urdu text correction system. Combine 4 ASR transcriptions into one perfect Urdu sentence.

Rules:
1. Use words where 2+ models agree (confidence >0.60)
2. Prefer highest confidence words when models disagree
3. Fix spelling errors
4. Ensure proper Urdu grammar
5. Use natural, simple Urdu language

Output: Only the corrected Urdu sentence with proper punctuation (۔). Use clean Urdu characters only.

Input:

"""

# Default prompt (fallback only - prompts are now provided from frontend)
# This is used only if no custom_prompt is sent from the frontend
STATIC_PROMPT_CONTEXT = """You are an expert Urdu text correction system. Combine 4 ASR transcriptions into one perfect Urdu sentence.

Rules:
1. Use words where 2+ models agree (confidence >0.60)
2. Prefer highest confidence words when models disagree
3. Fix spelling errors
4. Ensure proper Urdu grammar
5. Use natural, simple Urdu language

Output: Only the corrected Urdu sentence with proper punctuation (۔). Use clean Urdu characters only.

Input:

"""

In [ ]:
# Remove existing route if it exists (to allow re-running the cell)
# This prevents AssertionError when re-running the cell in Jupyter
if 'generate_correction' in app.view_functions:
    del app.view_functions['generate_correction']
    # Also remove from url_map if possible
    try:
        rules_to_remove = [rule for rule in app.url_map.iter_rules() 
                          if rule.endpoint == 'generate_correction']
        for rule in rules_to_remove:
            app.url_map._rules.remove(rule)
            if rule.endpoint in app.url_map._rules_by_endpoint:
                del app.url_map._rules_by_endpoint[rule.endpoint]
    except:
        pass  # If url_map cleanup fails, continue anyway

@app.route('/correct', methods=['POST'])
def generate_correction():
    try:
        # Check if model is loaded
        try:
            if chatbot is None:
                return jsonify({"error": "Model not loaded. Please load the model first."}), 500
        except NameError:
            return jsonify({"error": "Model not loaded. Please load the model first."}), 500
        
        data = request.json
        if not data:
            return jsonify({"error": "No JSON data received"}), 400
            
        hypotheses = data.get('hypotheses', [])
        custom_prompt = data.get('custom_prompt', '')

        if not hypotheses or len(hypotheses) != 4:
            return jsonify({"error": "Please provide exactly 4 hypotheses from ASR models."}), 400

        # Clear CUDA cache before inference to help prevent OOM
        try:
            import torch
            torch.cuda.empty_cache()
        except:
            pass

        # Construct the dynamic part of the prompt (hypotheses)
        dynamic_input = ""
        for i, hyp in enumerate(hypotheses):
            dynamic_input += f"H{i+1}: {hyp}\n"

        # Use custom prompt if provided, otherwise use default
        if custom_prompt and custom_prompt.strip():
            # If custom prompt contains {hypotheses} placeholder, replace it
            if "{hypotheses}" in custom_prompt:
                full_prompt = custom_prompt.replace("{hypotheses}", dynamic_input)
            else:
                # Otherwise, append hypotheses at the end
                full_prompt = f"{custom_prompt.strip()}\n\n{dynamic_input}"
        else:
            # Use default prompt
            full_prompt = f"{STATIC_PROMPT_CONTEXT}{dynamic_input}"
        
        # Ensure prompt doesn't end with just "Output:" - add a space or newline to encourage generation
        # Some models stop generating if the prompt ends with "Output:" without continuation
        if full_prompt.rstrip().endswith("Output:") or full_prompt.rstrip().endswith("Output"):
            full_prompt = full_prompt.rstrip() + "\n"
        
        # Also ensure the prompt ends properly to encourage generation
        if not full_prompt.endswith("\n") and not full_prompt.endswith(" "):
            full_prompt = full_prompt + " "

        # Debug: Print the prompt being sent to the model
        print("=" * 80)
        print("📤 PROMPT SENT TO MODEL:")
        print("=" * 80)
        print(full_prompt)
        print("=" * 80)
        print(f"Prompt length: {len(full_prompt)} characters")
        print("=" * 80)
        
        # Run Inference with optimized parameters
        # Adjusted to prevent empty responses
        # Note: Some parameters might not be supported by all pipeline versions
        try:
            response = chatbot(
                full_prompt,
                max_new_tokens=200,  # Increased to ensure we get output
                min_length=10,  # Force minimum output length
                do_sample=True,
                temperature=0.1,  # Slightly higher to encourage generation
                top_p=0.9,
                repetition_penalty=1.05,
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.eos_token_id  # Ensure proper padding
            )
        except TypeError as te:
            # If some parameters are not supported, try with basic parameters
            print(f"Warning: Some parameters not supported, using basic parameters. Error: {te}")
            try:
                response = chatbot(
                    full_prompt,
                    max_new_tokens=200,
                    min_length=10,
                    do_sample=True,
                    temperature=0.1,
                    top_p=0.9
                )
            except TypeError as te2:
                # Fallback to minimal parameters
                print(f"Warning: Advanced parameters not supported, using minimal. Error: {te2}")
                response = chatbot(
                    full_prompt,
                    max_new_tokens=200,
                    do_sample=True,
                    temperature=0.1
                )
        
        # Debug: Print the raw response structure
        print("=" * 80)
        print("📥 RAW RESPONSE STRUCTURE:")
        print("=" * 80)
        print(f"Response type: {type(response)}")
        print(f"Response: {response}")
        if isinstance(response, list) and len(response) > 0:
            print(f"Response[0] type: {type(response[0])}")
            print(f"Response[0] keys: {response[0].keys() if isinstance(response[0], dict) else 'N/A'}")
        print("=" * 80)

        if not response:
            return jsonify({"error": "Model returned None response"}), 500
            
        if not isinstance(response, list) or len(response) == 0:
            return jsonify({"error": f"Model returned invalid response format: {type(response)}"}), 500

        if "generated_text" not in response[0]:
            return jsonify({"error": f"Response missing 'generated_text' key. Response: {response[0]}"}), 500

        raw_response = response[0].get("generated_text", "").strip()
        
        # If we got an empty response, try with different parameters
        if not raw_response or len(raw_response.strip()) == 0:
            print("⚠️ First attempt returned empty. Trying with adjusted parameters...")
            try:
                # Try with more aggressive generation settings
                retry_response = chatbot(
                    full_prompt,
                    max_new_tokens=250,
                    min_length=15,
                    do_sample=True,
                    temperature=0.2,  # Higher temperature to encourage generation
                    top_p=0.95,
                    repetition_penalty=1.0,
                    pad_token_id=tokenizer.eos_token_id
                )
                retry_text = retry_response[0].get("generated_text", "").strip()
                if retry_text and len(retry_text) > 0:
                    print(f"✅ Retry successful! Got {len(retry_text)} characters")
                    raw_response = retry_text
                else:
                    print(f"⚠️ Retry also returned empty: {retry_response}")
            except Exception as retry_error:
                print(f"❌ Retry failed: {retry_error}")
        
        # Check if response is still empty after retry
        if not raw_response or len(raw_response.strip()) == 0:
            print("=" * 80)
            print("❌ EMPTY RESPONSE DEBUG:")
            print(f"   Full response object: {response}")
            print(f"   Response type: {type(response)}")
            if isinstance(response, list) and len(response) > 0:
                print(f"   Response[0] keys: {response[0].keys() if isinstance(response[0], dict) else 'N/A'}")
                print(f"   Response[0] full: {response[0]}")
            print(f"   Prompt length: {len(full_prompt)}")
            print(f"   Prompt preview: {full_prompt[:200]}...")
            print("=" * 80)
            return jsonify({
                "error": "Model returned empty text",
                "raw_response": str(response),
                "debug_info": f"The model generated an empty response. Prompt length: {len(full_prompt)} chars. This may indicate the model needs different parameters or the prompt format needs adjustment.",
                "prompt_preview": full_prompt[:300]
            }), 500
        
        # Check if response is just a prefix like "Answer:" with no content
        prefix_only_patterns = [
            r'^Answer:\s*$',
            r'^Response:\s*$',
            r'^Output:\s*$',
            r'^Corrected:\s*$',
        ]
        
        is_prefix_only = False
        for pattern in prefix_only_patterns:
            if re.match(pattern, raw_response, re.IGNORECASE):
                is_prefix_only = True
                break
        
        # If we only got a prefix with no content, return helpful error
        # (Don't retry as it causes CUDA OOM errors)
        if is_prefix_only or (len(raw_response) < 5 and not any(0x0600 <= ord(c) <= 0x06FF for c in raw_response)):
            print("⚠️ Model returned only a prefix or very short response with no Urdu content.")
            print(f"   Raw response: '{raw_response}'")
            return jsonify({
                "error": "Model returned only a prefix with no content",
                "raw_response": raw_response,
                "debug_info": f"The model generated only '{raw_response}' with no actual Urdu text. This may indicate the model needs different prompt parameters or the prompt format needs adjustment."
            }), 500
        
        if not raw_response or len(raw_response.strip()) == 0:
            return jsonify({
                "error": "Model returned empty text",
                "raw_response": str(response),
                "debug_info": "The model generated an empty response"
            }), 500
        
        # Debug: Print FULL raw response to help diagnose issues
        print("=" * 80)
        print("🔍 FULL RAW MODEL RESPONSE:")
        print("=" * 80)
        print(raw_response)
        print("=" * 80)
        print(f"Full response length: {len(raw_response)} characters")
        print(f"Full prompt length: {len(full_prompt)} characters")
        print("=" * 80)
        
        # IMPORTANT: The model returns the full prompt + generated text
        # We need to remove the prompt to get only the generated part
        corrected_text = raw_response
        
        # Remove the prompt from the response if it's present
        # The prompt typically ends with "Input:\n" or the hypotheses
        if full_prompt in corrected_text:
            corrected_text = corrected_text.replace(full_prompt, "", 1).strip()
            print(f"✅ Removed prompt using exact match. Remaining: {len(corrected_text)} chars")
        else:
            # Try to find where the prompt ends by looking for the last hypothesis marker
            # This is a fallback if exact match doesn't work
            found_marker = False
            for i in range(4, 0, -1):
                marker = f"H{i}:"
                if marker in corrected_text:
                    # Find the last occurrence and take everything after it
                    last_idx = corrected_text.rfind(marker)
                    if last_idx != -1:
                        # Find the end of this line
                        line_end = corrected_text.find('\n', last_idx)
                        if line_end != -1:
                            corrected_text = corrected_text[line_end:].strip()
                            print(f"✅ Removed prompt using marker H{i}. Remaining: {len(corrected_text)} chars")
                            found_marker = True
                            break
            if not found_marker:
                print(f"⚠️ Could not find prompt markers. Using full response: {len(corrected_text)} chars")
        
        # Remove common English prefixes that models add
        # But only if there's actual content after the prefix
        prefixes_to_remove = [
            "Answer:",
            "Response:",
            "### Response:",
            "### Answer:",
            "Output:",
            "Corrected text:",
            "Corrected:",
            "Result:",
        ]
        
        for prefix in prefixes_to_remove:
            # Only remove if there's content after the prefix
            if corrected_text.startswith(prefix):
                remaining = corrected_text[len(prefix):].strip()
                if len(remaining) > 0:  # Only remove if there's content after
                    corrected_text = remaining
                    print(f"✅ Removed prefix '{prefix}'. Remaining: {len(corrected_text)} chars")
                else:
                    print(f"⚠️ Prefix '{prefix}' found but no content after it. Keeping as-is.")
            elif prefix in corrected_text and not corrected_text.startswith(prefix):
                # If prefix appears in the middle, take everything after it
                parts = corrected_text.split(prefix, 1)
                if len(parts) > 1 and len(parts[-1].strip()) > 0:
                    corrected_text = parts[-1].strip()
                    print(f"✅ Removed prefix '{prefix}' from middle. Remaining: {len(corrected_text)} chars")
        
        print(f"📝 After prefix removal: '{corrected_text[:100]}...' (length: {len(corrected_text)})")
        
        # Take only the first line (the sentence) - but keep as fallback
        lines = corrected_text.split('\n')
        first_line = lines[0].strip() if lines else corrected_text.strip()
        print(f"📄 First line extracted: '{first_line[:100]}...' (length: {len(first_line)})")
        
        # Remove any trailing English words or explanations
        # Urdu sentences typically end with punctuation, so stop at first English word if present
        urdu_text = first_line
        # Find where Urdu text ends (if there's English after)
        for i, char in enumerate(first_line):
            if ord(char) < 128 and char.isalpha() and i > 10:  # Likely English after Urdu
                urdu_text = first_line[:i].strip()
                print(f"✂️ Removed trailing English. Urdu text: '{urdu_text[:100]}...'")
                break
        
        # Store less-cleaned version as fallback
        fallback_text = urdu_text.strip()
        print(f"💾 Fallback text stored: '{fallback_text[:100]}...' (length: {len(fallback_text)})")
        
        # Remove corrupted characters and encoding issues (light cleaning)
        corrected_text = re.sub(r'[*\\|]', '', urdu_text)
        
        # Remove any remaining confidence scores if model added them
        corrected_text = re.sub(r'\([\d.]+\)', '', corrected_text).strip()
        
        # Fix common corrupted Urdu characters
        char_fixes = {
            'ۃ': 'ہ',  # Corrupted heh to proper heh
            'ۓ': 'ے',  # Corrupted yeh to proper yeh
            'ۛ': '',   # Remove diacritic marks
            'ۘ': '',   # Remove diacritic marks
            '۟': '',   # Remove diacritic marks
            'ۙ': '',   # Remove diacritic marks
            'ۚ': '',   # Remove diacritic marks
        }
        for corrupted, correct in char_fixes.items():
            corrected_text = corrected_text.replace(corrupted, correct)
        
        # Count Urdu characters to validate content
        urdu_char_count = sum(1 for char in corrected_text if 0x0600 <= ord(char) <= 0x06FF)
        
        # Less aggressive cleaning - keep more characters
        # Only remove clearly problematic characters, keep everything else
        cleaned_chars = []
        for char in corrected_text:
            code_point = ord(char)
            # Keep Urdu/Arabic characters (0600-06FF), spaces, and punctuation
            if (0x0600 <= code_point <= 0x06FF) or char.isspace() or char in '۔،؟!.,;:!?':
                cleaned_chars.append(char)
            # Keep digits (might be part of text)
            elif char.isdigit():
                cleaned_chars.append(char)
            # Remove only clearly problematic characters (control chars, special symbols)
            elif code_point >= 32 and code_point < 127:  # Printable ASCII
                # Keep common punctuation, remove special symbols
                if char in '.,;:!?()[]{}':
                    cleaned_chars.append(char)
        
        corrected_text = ''.join(cleaned_chars)
        
        # Normalize Unicode (combine diacritics properly)
        corrected_text = unicodedata.normalize('NFC', corrected_text)
        
        # Clean up extra spaces
        corrected_text = ' '.join(corrected_text.split())
        
        # Re-count Urdu characters after cleaning
        urdu_char_count_after = sum(1 for char in corrected_text if 0x0600 <= ord(char) <= 0x06FF)
        
        # Validation: Check if we have meaningful Urdu content
        # If aggressive cleaning removed too much, use fallback
        if urdu_char_count_after < 2 and urdu_char_count > 0:
            # Use fallback (less cleaned version)
            print(f"Warning: Aggressive cleaning removed too much. Using fallback.")
            corrected_text = fallback_text
            # Light cleaning on fallback
            corrected_text = re.sub(r'[*\\|]', '', corrected_text)
            corrected_text = re.sub(r'\([\d.]+\)', '', corrected_text).strip()
            for corrupted, correct in char_fixes.items():
                corrected_text = corrected_text.replace(corrupted, correct)
            corrected_text = unicodedata.normalize('NFC', corrected_text)
            corrected_text = ' '.join(corrected_text.split())
            urdu_char_count_after = sum(1 for char in corrected_text if 0x0600 <= ord(char) <= 0x06FF)
        
        # Final validation - need at least some Urdu content or reasonable length
        print("=" * 80)
        print("📊 FINAL VALIDATION:")
        print(f"   Cleaned text length: {len(corrected_text)}")
        print(f"   Urdu character count: {urdu_char_count_after}")
        print(f"   Cleaned text: '{corrected_text[:200]}...'")
        print("=" * 80)
        
        if len(corrected_text) < 2 or (urdu_char_count_after == 0 and len(corrected_text) < 5):
            print("❌ VALIDATION FAILED: Response too short or no Urdu content")
            print(f"   Raw response (first 1000 chars): {raw_response[:1000]}")
            print(f"   After prompt removal: {corrected_text[:500]}")
            return jsonify({
                "error": "Model returned corrupted or empty response",
                "raw_response": raw_response,  # Full response for debugging
                "cleaned_length": len(corrected_text),
                "urdu_char_count": urdu_char_count_after,
                "cleaned_text_preview": corrected_text[:200],
                "debug_info": f"After cleaning: length={len(corrected_text)}, urdu_chars={urdu_char_count_after}. Raw response length: {len(raw_response)}"
            }), 500
        
        print("✅ VALIDATION PASSED: Returning cleaned text")
        print("=" * 80)

        return jsonify({
            "status": "success",
            "corrected_text": corrected_text
        })

    except RuntimeError as e:
        # Handle CUDA out of memory errors specifically
        error_str = str(e)
        if "CUDA out of memory" in error_str or "out of memory" in error_str.lower():
            import traceback
            error_details = traceback.format_exc()
            print(f"❌ CUDA Out of Memory Error: {error_details}")
            
            # Try to clear CUDA cache
            try:
                import torch
                torch.cuda.empty_cache()
                print("✅ Cleared CUDA cache")
            except:
                pass
            
            return jsonify({
                "error": "CUDA out of memory",
                "error_type": "RuntimeError",
                "details": error_str,
                "suggestion": "The GPU ran out of memory. Try reducing max_new_tokens or restart the kernel to free GPU memory."
            }), 500
        else:
            # Other RuntimeErrors
            import traceback
            error_details = traceback.format_exc()
            print(f"Error in generate_correction: {error_details}")
            return jsonify({
                "error": str(e),
                "error_type": type(e).__name__,
                "details": error_details
            }), 500
    except Exception as e:
        import traceback
        error_details = traceback.format_exc()
        print(f"Error in generate_correction: {error_details}")
        return jsonify({
            "error": str(e),
            "error_type": type(e).__name__,
            "details": error_details
        }), 500

In [ ]:
# ==========================================
# DEBUG: Display Last Model Response
# ==========================================
# Run this cell after making a request to see what the model returned
# This helps debug issues with model responses

# Store the last response for debugging
last_model_response = None
last_full_prompt = None

# You can also manually test the model here:
# test_hypotheses = ["test1", "test2", "test3", "test4"]
# test_prompt = "Your prompt here"
# test_response = chatbot(test_prompt, max_new_tokens=200, do_sample=True, temperature=0.05)
# print("Model Response:", test_response)

print("💡 To see the last model response, check the Flask server output above.")
print("💡 Or make a request from the frontend and check the console output.")


In [ ]:
@app.route('/', methods=['GET'])
def health_check():
    return jsonify({"status": "CORAL Backend is Running"}), 200


In [ ]:
# ==========================================
# RUN SERVER WITH NGROK
# ==========================================
def run_app():
    app.run(port=5000)

# Set Auth Token
if NGROK_AUTH_TOKEN == "NGROK_AUTH_TOKEN":
    print("❌ ERROR: You forgot to set your Ngrok Auth Token at the top of the script!")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(5000).public_url
    print(f"\n🚀 BACKEND IS LIVE! Copy this URL for your Frontend:\n")
    print(f"👉 {public_url} 👈\n")

    # Start Flask in a separate thread
    threading.Thread(target=run_app).start()